In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import shutil

In [3]:
!mkdir /content/covidctmd/
!mkdir /content/covidctmd/covidctmd_covid_part1
!mkdir /content/covidctmd/covidctmd_covid_part2
!mkdir /content/covidctmd/covidctmd_covid_part3
!mkdir /content/covidctmd/covidctmd_normal_part1
!mkdir /content/covidctmd/covidctmd_normal_part2
!mkdir /content/covidctmd/covidctmd_normal_part3

In [4]:
!mkdir /content/split_folds/

In [5]:
!unzip -q /content/drive/MyDrive/split_folds.zip -d /content/split_folds/

In [6]:
!unzip -q /content/drive/MyDrive/covidctmd_covid_segmented_part1.zip -d /content/covidctmd/covidctmd_covid_part1/

In [7]:
!unzip -q /content/drive/MyDrive/covidctmd_covid_segmented_part2.zip -d /content/covidctmd/covidctmd_covid_part2/

In [8]:
!unzip -q /content/drive/MyDrive/covidctmd_covid_segmented_part3.zip -d /content/covidctmd/covidctmd_covid_part3/

In [9]:
!unzip -q /content/drive/MyDrive/covidctmd_normal_segmented_part1.zip -d /content/covidctmd/covidctmd_normal_part1/

In [10]:
!unzip -q /content/drive/MyDrive/covidctmd_normal_segmented_part2.zip -d /content/covidctmd/covidctmd_normal_part2/

In [11]:
!unzip -q /content/drive/MyDrive/covidctmd_normal_segmented_part3.zip -d /content/covidctmd/covidctmd_normal_part3/

In [12]:
!mkdir /content/model_weights

In [13]:
!unzip -q /content/drive/MyDrive/split_seg_densenet121.zip -d /content/model_weights/

In [14]:
#import kerastuner as kt
from tensorflow import keras
import tensorflow as tf
#from kerastuner.tuners import RandomSearch
#from kerastuner.engine.hyperparameters import HyperParameter as hp
from keras.layers import Dense,Dropout,Activation,Add,MaxPooling2D,Conv2D,Flatten
from keras.models import Sequential
from keras.preprocessing.image import ImageDataGenerator
import numpy as np
import os
import matplotlib.pyplot as plt
from tensorflow.keras.applications import VGG19
from keras import layers
from keras.preprocessing import image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.model_selection import train_test_split
import datetime
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.metrics import precision_recall_fscore_support as score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import itertools
import cv2
from sklearn.metrics import precision_recall_fscore_support as score
from tensorflow.keras.models import Model
!pip install imutils
import imutils
import matplotlib as mpl
import time
from scipy import interp
from sklearn.metrics import roc_curve,auc
from sklearn import metrics
from tensorflow.keras.applications.inception_v3 import preprocess_input
import math
import shutil
!pip install openpyxl

In [15]:
def create_model(img_size,model_name,channels,classes,activation,weights,lr,crossentropy,optimizer_hyperparams,beta_1,beta_2,epsilon,decay,fine_tuning,unfrozen_layers):

    if model_name=='InceptionV3':
        base_model = tf.keras.applications.InceptionV3(include_top = False, weights = weights, input_shape = (img_size, img_size, channels))
    elif model_name=='InceptionResNetV2':
        base_model = tf.keras.applications.InceptionResNetV2(include_top = False, weights = weights, input_shape = (img_size, img_size, channels))
    elif model_name=='Xception':
        base_model = tf.keras.applications.Xception(include_top = False, weights = weights, input_shape = (img_size, img_size, channels))
    elif model_name=='DenseNet121':
        base_model = tf.keras.applications.DenseNet121(include_top = False, weights = weights, input_shape = (img_size, img_size, channels))
    elif model_name=='DenseNet169':
        base_model = tf.keras.applications.DenseNet169(include_top = False, weights = weights, input_shape = (img_size, img_size, channels))
    elif model_name=='DenseNet201':
        base_model = tf.keras.applications.DenseNet201(include_top = False, weights = weights, input_shape = (img_size, img_size, channels))
    elif model_name=='ResNet50':
        base_model = tf.keras.applications.ResNet50(include_top = False, weights = weights, input_shape = (img_size, img_size, channels))
    elif model_name=='ResNet101':
        base_model = tf.keras.applications.ResNet101(include_top = False, weights = weights, input_shape = (img_size, img_size, channels))
    elif model_name=='ResNet152':
        base_model = tf.keras.applications.ResNet152(include_top = False, weights = weights, input_shape = (img_size, img_size, channels))
    elif model_name=='MobileNetV2':
        base_model = tf.keras.applications.MobileNetV2(include_top = False, weights = weights, input_shape = (img_size, img_size, channels))
    elif model_name=='VGG16':
        base_model = tf.keras.applications.VGG16(include_top = False, weights = weights, input_shape = (img_size, img_size, channels))

    base_model.trainable=True

    x=base_model.output
    if model_name=='VGG16':
        x=tf.keras.layers.Flatten()(x)
    else:
        x=tf.keras.layers.GlobalAveragePooling2D()(x)
    #x=tf.keras.layers.Dropout(0.5)(x)
    out = tf.keras.layers.Dense(classes, activation = 'softmax')(x)


    # Build the Keras model

    model = tf.keras.models.Model(inputs = base_model.input, outputs = out)
    if optimizer_hyperparams==True:
        print('Optimizer Hyperparams: True')
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr, beta_1=beta_1, beta_2=beta_2, epsilon=epsilon, decay=decay)
    else:
        print('Optimizer Hyperparams: False')
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

    model.compile(loss=crossentropy, optimizer=optimizer, metrics=['accuracy'])
    return model

In [16]:
def plot_confusion_matrix_concat_test(val_total_cm, MODEL_NAME, foldno, CLASS1, CLASS2):

    with plt.style.context('default'):

        plt.figure()

        ax = sns.heatmap(val_total_cm, annot=True, fmt="d", cmap='Blues')

        ax.set_title('Confusion Matrix for '+MODEL_NAME);
        ax.set_xlabel('\nPredicted')
        ax.set_ylabel('True');

        ## Ticket labels - List must be in alphabetical order
        ax.xaxis.set_ticklabels(['Covid-19','Normal'])
        ax.yaxis.set_ticklabels(['Covid-19','Normal'])

        ## Display the visualization of the Confusion Matrix.
        plt.savefig('/content/'+MODEL_NAME+'/confusion_matrix/cm_concat_test_decimal.png')
        plt.show()

        ax = sns.heatmap(val_total_cm/np.sum(val_total_cm), annot=True,
            fmt='.2%', cmap='Blues')

        ax.set_title('Confusion Matrix for '+MODEL_NAME);
        ax.set_xlabel('\nPredicted')
        ax.set_ylabel('True');

        ## Ticket labels - List must be in alphabetical order
        ax.xaxis.set_ticklabels(['Covid-19','Normal'])
        ax.yaxis.set_ticklabels(['Covid-19','Normal'])

        ## Display the visualization of the Confusion Matrix.
        plt.savefig('/content/'+MODEL_NAME+'/confusion_matrix/cm_concat_test_percentage.png')
        plt.show()

        group_counts = ["{0:0.0f}".format(value) for value in
                    val_total_cm.flatten()]
        group_percentages = ["{0:.2%}".format(value) for value in
                         val_total_cm.flatten()/np.sum(val_total_cm)]

        labels = [f"{v1}\n{v2}" for v1, v2 in
              zip(group_counts,group_percentages)]
        labels = np.asarray(labels).reshape(2,2)
        ax=sns.heatmap(val_total_cm, annot=labels, fmt='', cmap='Blues')

        ax.set_title('Confusion Matrix for '+MODEL_NAME)
        ax.set_xlabel('\nPredicted')
        ax.set_ylabel('True');

        ## Ticket labels - List must be in alphabetical order
        ax.xaxis.set_ticklabels(['Covid-19','Normal'])
        ax.yaxis.set_ticklabels(['Covid-19','Normal'])

        ## Display the visualization of the Confusion Matrix.
        plt.savefig('/content/'+MODEL_NAME+'/confusion_matrix/cm_concat_test_all.png')
        plt.show()

In [17]:
def plot_confusion_matrix_concat_val(val_total_cm, MODEL_NAME, foldno, CLASS1, CLASS2):

    with plt.style.context('default'):

        plt.figure()

        ax = sns.heatmap(val_total_cm, annot=True, fmt="d", cmap='Blues')

        ax.set_title('Confusion Matrix for '+MODEL_NAME);
        ax.set_xlabel('\nPredicted')
        ax.set_ylabel('True');

        ## Ticket labels - List must be in alphabetical order
        ax.xaxis.set_ticklabels(['Covid-19','Normal'])
        ax.yaxis.set_ticklabels(['Covid-19','Normal'])

        ## Display the visualization of the Confusion Matrix.
        plt.savefig('/content/'+MODEL_NAME+'/confusion_matrix/cm_concat_val_decimal.png')
        plt.show()

        ax = sns.heatmap(val_total_cm/np.sum(val_total_cm), annot=True,
            fmt='.2%', cmap='Blues')

        ax.set_title('Confusion Matrix for '+MODEL_NAME);
        ax.set_xlabel('\nPredicted')
        ax.set_ylabel('True');

        ## Ticket labels - List must be in alphabetical order
        ax.xaxis.set_ticklabels(['Covid-19','Normal'])
        ax.yaxis.set_ticklabels(['Covid-19','Normal'])

        ## Display the visualization of the Confusion Matrix.
        plt.savefig('/content/'+MODEL_NAME+'/confusion_matrix/cm_concat_val_percentage.png')
        plt.show()

        group_counts = ["{0:0.0f}".format(value) for value in
                    val_total_cm.flatten()]
        group_percentages = ["{0:.2%}".format(value) for value in
                         val_total_cm.flatten()/np.sum(val_total_cm)]

        labels = [f"{v1}\n{v2}" for v1, v2 in
              zip(group_counts,group_percentages)]
        labels = np.asarray(labels).reshape(2,2)
        ax=sns.heatmap(val_total_cm, annot=labels, fmt='', cmap='Blues')

        ax.set_title('Confusion Matrix for '+MODEL_NAME)
        ax.set_xlabel('\nPredicted')
        ax.set_ylabel('True');

        ## Ticket labels - List must be in alphabetical order
        ax.xaxis.set_ticklabels(['Covid-19','Normal'])
        ax.yaxis.set_ticklabels(['Covid-19','Normal'])

        ## Display the visualization of the Confusion Matrix.
        plt.savefig('/content/'+MODEL_NAME+'/confusion_matrix/cm_concat_val_all.png')
        plt.show()

In [18]:
def plot_confusion_matrix_test(cm, MODEL_NAME, foldno, CLASS1, CLASS2):

    with plt.style.context('default'):

        plt.figure()

        ax = sns.heatmap(cm, annot=True, fmt="d", cmap='Blues')

        ax.set_title('Confusion Matrix for '+MODEL_NAME+' \n Fold '+str(foldno));
        ax.set_xlabel('\nPredicted')
        ax.set_ylabel('True');

        ## Ticket labels - List must be in alphabetical order
        ax.xaxis.set_ticklabels(['Covid-19','Normal'])
        ax.yaxis.set_ticklabels(['Covid-19','Normal'])

        ## Display the visualization of the Confusion Matrix.
        plt.savefig('/content/'+MODEL_NAME+'/confusion_matrix/cm_test_decimal'+str(foldno)+'.png')
        plt.show()

        ax = sns.heatmap(cm/np.sum(cm), annot=True,
                fmt='.2%', cmap='Blues')

        ax.set_title('Confusion Matrix for '+MODEL_NAME+' \n Fold '+str(foldno));
        ax.set_xlabel('\nPredicted')
        ax.set_ylabel('True');

        ## Ticket labels - List must be in alphabetical order
        ax.xaxis.set_ticklabels(['Covid-19','Normal'])
        ax.yaxis.set_ticklabels(['Covid-19','Normal'])

        ## Display the visualization of the Confusion Matrix.
        plt.savefig('/content/'+MODEL_NAME+'/confusion_matrix/cm_test_percentage'+str(foldno)+'.png')
        plt.show()

        group_counts = ["{0:0.0f}".format(value) for value in
                cm.flatten()]
        group_percentages = ["{0:.2%}".format(value) for value in
                         cm.flatten()/np.sum(cm)]

        labels = [f"{v1}\n{v2}" for v1, v2 in
              zip(group_counts,group_percentages)]
        labels = np.asarray(labels).reshape(2,2)
        ax=sns.heatmap(cm, annot=labels, fmt='', cmap='Blues')

        ax.set_title('Confusion Matrix for '+MODEL_NAME+' \n Fold '+str(foldno));
        ax.set_xlabel('\nPredicted')
        ax.set_ylabel('True');

        ## Ticket labels - List must be in alphabetical order
        ax.xaxis.set_ticklabels(['Covid-19','Normal'])
        ax.yaxis.set_ticklabels(['Covid-19','Normal'])

        ## Display the visualization of the Confusion Matrix.
        plt.savefig('/content/'+MODEL_NAME+'/confusion_matrix/cm_test_all'+str(foldno)+'.png')
        plt.show()

In [19]:
def plot_confusion_matrix_val(cm, MODEL_NAME, foldno, CLASS1, CLASS2):

    with plt.style.context('default'):

        plt.figure()

        ax = sns.heatmap(cm, annot=True, fmt="d", cmap='Blues')

        ax.set_title('Confusion Matrix for '+MODEL_NAME+' \n Fold '+str(foldno));
        ax.set_xlabel('\nPredicted')
        ax.set_ylabel('True');

        ## Ticket labels - List must be in alphabetical order
        ax.xaxis.set_ticklabels(['Covid-19','Normal'])
        ax.yaxis.set_ticklabels(['Covid-19','Normal'])

        ## Display the visualization of the Confusion Matrix.
        plt.savefig('/content/'+MODEL_NAME+'/confusion_matrix/cm_val_decimal'+str(foldno)+'.png')
        plt.show()

        ax = sns.heatmap(cm/np.sum(cm), annot=True,
                fmt='.2%', cmap='Blues')

        ax.set_title('Confusion Matrix for '+MODEL_NAME+' \n Fold '+str(foldno));
        ax.set_xlabel('\nPredicted')
        ax.set_ylabel('True');

        ## Ticket labels - List must be in alphabetical order
        ax.xaxis.set_ticklabels(['Covid-19','Normal'])
        ax.yaxis.set_ticklabels(['Covid-19','Normal'])

        ## Display the visualization of the Confusion Matrix.
        plt.savefig('/content/'+MODEL_NAME+'/confusion_matrix/cm_val_percentage'+str(foldno)+'.png')
        plt.show()

        group_counts = ["{0:0.0f}".format(value) for value in
                    cm.flatten()]
        group_percentages = ["{0:.2%}".format(value) for value in
                         cm.flatten()/np.sum(cm)]

        labels = [f"{v1}\n{v2}" for v1, v2 in
              zip(group_counts,group_percentages)]
        labels = np.asarray(labels).reshape(2,2)
        ax=sns.heatmap(cm, annot=labels, fmt='', cmap='Blues')

        ax.set_title('Confusion Matrix for '+MODEL_NAME+' \n Fold '+str(foldno));
        ax.set_xlabel('\nPredicted')
        ax.set_ylabel('True');

        ## Ticket labels - List must be in alphabetical order
        ax.xaxis.set_ticklabels(['Covid-19','Normal'])
        ax.yaxis.set_ticklabels(['Covid-19','Normal'])

        ## Display the visualization of the Confusion Matrix.
        plt.savefig('/content/'+MODEL_NAME+'/confusion_matrix/cm_val_all'+str(foldno)+'.png')
        plt.show()

In [20]:
def plot_roc_curve_val_labels(fpr,tpr,foldno,roc_auc,MODEL_NAME):

    with plt.style.context('default'):

        plt.figure()
        plt.plot(fpr, tpr, lw=2, alpha=0.8, color='b', label='ROC fold %d (AUC = %0.2f)' % (foldno, roc_auc))
        plt.plot([0,1],[0,1],linestyle = '--',lw = 2,color = 'r',label="Chance",alpha=0.8)
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Receiver Operating Characteristic for '+MODEL_NAME+" \n Fold "+str(foldno))
        plt.legend(loc="lower right")
        plt.savefig('/content/'+MODEL_NAME+'/roc_curve_val/validation_roc_curve_labels'+str(foldno)+'.png')
        plt.show()
        plt.close()

In [21]:
def plot_roc_curve_test(fpr,tpr,foldno,roc_auc,MODEL_NAME):

    with plt.style.context('default'):

        plt.figure()
        plt.plot(fpr, tpr, lw=2, alpha=0.8, color='b', label='ROC fold %d (AUC = %0.2f)' % (foldno, roc_auc))
        plt.plot([0,1],[0,1],linestyle = '--',lw = 2,color = 'r',label="Chance",alpha=0.8)
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Receiver Operating Characteristic for '+MODEL_NAME+" \n Fold "+str(foldno))
        plt.legend(loc="lower right")
        plt.savefig('/content/'+MODEL_NAME+'/roc_curve_test/test_roc_curve_'+str(foldno)+'.png')
        plt.show()
        plt.close()

In [22]:
def plot_roc_curve_val_probs(fpr,tpr,foldno,roc_auc,MODEL_NAME):

    with plt.style.context('default'):

        plt.figure()
        plt.plot(fpr, tpr, lw=2, alpha=0.8, color='b', label='ROC fold %d (AUC = %0.2f)' % (foldno, roc_auc))
        plt.plot([0,1],[0,1],linestyle = '--',lw = 2,color = 'r',label="Chance",alpha=0.8)
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Receiver Operating Characteristic for '+MODEL_NAME+" \n Fold "+str(foldno))
        plt.legend(loc="lower right")
        plt.savefig('/content/'+MODEL_NAME+'/roc_curve_val/validation_roc_curve_probs'+str(foldno)+'.png')
        plt.show()
        plt.close()

In [23]:
N_SPLITS=5
FOLDNO=4
IMG_SIZE=256
CHANNELS=3
LEARNING_RATE=1e-6
BATCH_SIZE=32
CLASSES=2
CLASS1="Covid-19"
CLASS2="Normal"
EPOCHS=20
CLASS_MODE='categorical'
CROSSENTROPY='categorical_crossentropy'
MODEL_NAME="DenseNet121"
OPTIMIZER="Adam"
ACTIVATION="softmax"
WEIGHTS="imagenet"
OPTIMIZER_HYPERPARAMS=False
BETA_1=0.9
BETA_2=0.999
EPSILON=0.1
DECAY=0.0
ANNEALER=False
ANNEALER_MONITOR='val_loss'
ANNEALER_FACTOR=0.1
ANNEALER_PATIENCE=3
EARLY=False
EARLY_MONITOR='val_loss'
EARLY_PATIENCE=10
MIN_DELTA=0.001
RESCALE=True
SAMPLEWISE_CENTER=False
SAMPLEWISE_STD_NORM=False
AUGMENTATION=True
ROTATION_RANGE=15
WIDTH_SHIFT_RANGE=0.15
HEIGHT_SHIFT_RANGE=0.15
ZOOM_RANGE=None
HORIZONTAL_FLIP=True
VERTICAL_FLIP=False
TEST_SIZE=0.1666
FINE_TUNING=False
UNFROZEN_LAYERS=0
POOLING='None'
SEED=42
NORMALIZED='Normalized'
SPLIT='Split'
SEGMENTED='Segmented'

In [24]:
train_df = pd.read_csv('/content/drive/MyDrive/train_COVIDx_CT-3A.txt', sep=" ", header=None)
train_df.columns=['filename', 'label', 'xmin','ymin','xmax','ymax']
train_df=train_df.drop(['xmin', 'ymin','xmax', 'ymax'], axis=1 )

val_df = pd.read_csv('/content/drive/MyDrive/val_COVIDx_CT-3A.txt', sep=" ", header=None)
val_df.columns=['filename', 'label', 'xmin','ymin','xmax','ymax']
val_df=val_df.drop(['xmin', 'ymin','xmax', 'ymax'], axis=1 )

test_df = pd.read_csv('/content/drive/MyDrive/test_COVIDx_CT-3A.txt', sep=" ", header=None)
test_df.columns=['filename', 'label', 'xmin','ymin','xmax','ymax']
test_df=test_df.drop(['xmin', 'ymin','xmax', 'ymax'], axis=1 )

In [25]:
train=pd.concat([train_df,val_df,test_df])

In [26]:
train_normal=train[train['filename'].str.startswith('COVIDCTMD-normal')]
normal=train_normal['filename'].str.split('-',n=2, expand=True)
normal[0]=normal[0]+'-'+normal[1]
normal=normal.drop(columns=[1])
normal.rename(columns={0: 'patient_id', 2: 'images'}, inplace=True)
normal['target']=CLASS2
normal=normal.drop(columns=['images'])
normal['images']=train_normal['filename']

In [27]:
covid_dir1='/content/covidctmd/covidctmd_covid_part1/covidctmd_covid_segmented_part1/'
covid_dir2='/content/covidctmd/covidctmd_covid_part2/covidctmd_covid_segmented_part2/'
covid_dir3='/content/covidctmd/covidctmd_covid_part3/covidctmd_covid_segmented_part3/'
covidlist1=os.listdir(covid_dir1)
covidlist2=os.listdir(covid_dir2)
covidlist3=os.listdir(covid_dir3)
covidlist=covidlist1+covidlist2+covidlist3
covidlist.sort()

In [28]:
normal_dir1='/content/covidctmd/covidctmd_normal_part1/'
normal_dir2='/content/covidctmd/covidctmd_normal_part2/'
normal_dir3='/content/covidctmd/covidctmd_normal_part3/'
normallist1=os.listdir(normal_dir1)
normallist2=os.listdir(normal_dir2)
normallist3=os.listdir(normal_dir3)
normallist=normallist1+normallist2+normallist3
normallist.sort()

In [29]:
covid_df1=pd.DataFrame({'images': covidlist1, 'target': CLASS1})
covid_df1['image_path'] = covid_dir1 + covid_df1['images']
covid_df2=pd.DataFrame({'images': covidlist2, 'target': CLASS1})
covid_df2['image_path'] = covid_dir2 + covid_df2['images']
covid_df3=pd.DataFrame({'images': covidlist3, 'target': CLASS1})
covid_df3['image_path'] = covid_dir3 + covid_df3['images']
covid_df=pd.concat([covid_df1, covid_df2, covid_df3])
covid_patient_df=pd.DataFrame()
covid_patient_df['patient_id']=covid_df.images.str.rsplit("-", n=1, expand = True)[0]
final_covid_df=pd.concat([covid_patient_df, covid_df], axis=1)
covid_pat_unique=pd.DataFrame(final_covid_df['patient_id'].unique())

In [30]:
normal_df1=pd.DataFrame({'images': normallist1, 'target': CLASS2})
normal_df1['image_path'] = normal_dir1 + normal_df1['images']
normal_df2=pd.DataFrame({'images': normallist2, 'target': CLASS2})
normal_df2['image_path'] = normal_dir2 + normal_df2['images']
normal_df3=pd.DataFrame({'images': normallist3, 'target': CLASS2})
normal_df3['image_path'] = normal_dir3 + normal_df3['images']
normal_df=pd.concat([normal_df1, normal_df2, normal_df3])
normal_patient_df=pd.DataFrame()
normal_patient_df['patient_id']=normal_df.images.str.rsplit("_", n=1, expand = True)[0]
final_normal_df=pd.concat([normal_patient_df, normal_df], axis=1)
normal_pat_unique=pd.DataFrame(final_normal_df['patient_id'].unique())

In [31]:
final_normal_df['images']=final_normal_df['images'].str.replace(r'G', '', regex=True)
final_normal_df['images']=final_normal_df['images'].str.replace(r'_', '-', regex=True)
normal['images']=normal['images'].str[:-3]+'tif'
new_normal_df=final_normal_df[final_normal_df['images'].isin(normal['images'])]
final_normal_df=new_normal_df

In [32]:
file = open("/content/split_folds/train_covid_folds1.txt", "r")
data = file.read()

train_covid_folds1 = data.split("\n")
file.close()
train_covid_folds1.pop()

file = open("/content/split_folds/train_covid_folds2.txt", "r")
data = file.read()

train_covid_folds2 = data.split("\n")
file.close()
train_covid_folds2.pop()

file = open("/content/split_folds/train_covid_folds3.txt", "r")
data = file.read()

train_covid_folds3 = data.split("\n")
file.close()
train_covid_folds3.pop()

file = open("/content/split_folds/train_covid_folds4.txt", "r")
data = file.read()

train_covid_folds4 = data.split("\n")
file.close()
train_covid_folds4.pop()

file = open("/content/split_folds/train_covid_folds5.txt", "r")
data = file.read()

train_covid_folds5 = data.split("\n")
file.close()
train_covid_folds5.pop()

''

In [33]:
file = open("/content/split_folds/valid_covid_folds1.txt", "r")
data = file.read()

valid_covid_folds1 = data.split("\n")
file.close()
valid_covid_folds1.pop()

file = open("/content/split_folds/valid_covid_folds2.txt", "r")
data = file.read()

valid_covid_folds2 = data.split("\n")
file.close()
valid_covid_folds2.pop()

file = open("/content/split_folds/valid_covid_folds3.txt", "r")
data = file.read()

valid_covid_folds3 = data.split("\n")
file.close()
valid_covid_folds3.pop()

file = open("/content/split_folds/valid_covid_folds4.txt", "r")
data = file.read()

valid_covid_folds4 = data.split("\n")
file.close()
valid_covid_folds4.pop()

file = open("/content/split_folds/valid_covid_folds5.txt", "r")
data = file.read()

valid_covid_folds5 = data.split("\n")
file.close()
valid_covid_folds5.pop()

''

In [34]:
file = open("/content/split_folds/train_normal_folds1.txt", "r")
data = file.read()

train_normal_folds1 = data.split("\n")
file.close()
train_normal_folds1.pop()

file = open("/content/split_folds/train_normal_folds2.txt", "r")
data = file.read()

train_normal_folds2 = data.split("\n")
file.close()
train_normal_folds2.pop()

file = open("/content/split_folds/train_normal_folds3.txt", "r")
data = file.read()

train_normal_folds3 = data.split("\n")
file.close()
train_normal_folds3.pop()

file = open("/content/split_folds/train_normal_folds4.txt", "r")
data = file.read()

train_normal_folds4 = data.split("\n")
file.close()
train_normal_folds4.pop()

file = open("/content/split_folds/train_normal_folds5.txt", "r")
data = file.read()

train_normal_folds5 = data.split("\n")
file.close()
train_normal_folds5.pop()

''

In [35]:
file = open("/content/split_folds/valid_normal_folds1.txt", "r")
data = file.read()

valid_normal_folds1 = data.split("\n")
file.close()
valid_normal_folds1.pop()

file = open("/content/split_folds/valid_normal_folds2.txt", "r")
data = file.read()

valid_normal_folds2 = data.split("\n")
file.close()
valid_normal_folds2.pop()

file = open("/content/split_folds/valid_normal_folds3.txt", "r")
data = file.read()

valid_normal_folds3 = data.split("\n")
file.close()
valid_normal_folds3.pop()

file = open("/content/split_folds/valid_normal_folds4.txt", "r")
data = file.read()

valid_normal_folds4 = data.split("\n")
file.close()
valid_normal_folds4.pop()

file = open("/content/split_folds/valid_normal_folds5.txt", "r")
data = file.read()

valid_normal_folds5 = data.split("\n")
file.close()
valid_normal_folds5.pop()

''

In [36]:
new_train_covid_fold1=final_covid_df[final_covid_df['patient_id'].isin(train_covid_folds1)]
new_train_covid_fold2=final_covid_df[final_covid_df['patient_id'].isin(train_covid_folds2)]
new_train_covid_fold3=final_covid_df[final_covid_df['patient_id'].isin(train_covid_folds3)]
new_train_covid_fold4=final_covid_df[final_covid_df['patient_id'].isin(train_covid_folds4)]
new_train_covid_fold5=final_covid_df[final_covid_df['patient_id'].isin(train_covid_folds5)]
#new_train_covid_fold6=covid[covid['patient_id'].isin(train_covid_folds[5][0])]
#new_train_covid_fold7=covid[covid['patient_id'].isin(train_covid_folds[6][0])]
#new_train_covid_fold8=covid[covid['patient_id'].isin(train_covid_folds[7][0])]
#new_train_covid_fold9=covid[covid['patient_id'].isin(train_covid_folds[8][0])]
#new_train_covid_fold10=covid[covid['patient_id'].isin(train_covid_folds[9][0])]

In [37]:
new_train_normal_fold1=final_normal_df[final_normal_df['patient_id'].isin(train_normal_folds1)]
new_train_normal_fold2=final_normal_df[final_normal_df['patient_id'].isin(train_normal_folds2)]
new_train_normal_fold3=final_normal_df[final_normal_df['patient_id'].isin(train_normal_folds3)]
new_train_normal_fold4=final_normal_df[final_normal_df['patient_id'].isin(train_normal_folds4)]
new_train_normal_fold5=final_normal_df[final_normal_df['patient_id'].isin(train_normal_folds5)]
#new_train_covid_fold6=covid[covid['patient_id'].isin(train_covid_folds[5][0])]
#new_train_covid_fold7=covid[covid['patient_id'].isin(train_covid_folds[6][0])]
#new_train_covid_fold8=covid[covid['patient_id'].isin(train_covid_folds[7][0])]
#new_train_covid_fold9=covid[covid['patient_id'].isin(train_covid_folds[8][0])]
#new_train_covid_fold10=covid[covid['patient_id'].isin(train_covid_folds[9][0])]

In [38]:
new_valid_covid_fold1=final_covid_df[final_covid_df['patient_id'].isin(valid_covid_folds1)]
new_valid_covid_fold2=final_covid_df[final_covid_df['patient_id'].isin(valid_covid_folds2)]
new_valid_covid_fold3=final_covid_df[final_covid_df['patient_id'].isin(valid_covid_folds3)]
new_valid_covid_fold4=final_covid_df[final_covid_df['patient_id'].isin(valid_covid_folds4)]
new_valid_covid_fold5=final_covid_df[final_covid_df['patient_id'].isin(valid_covid_folds5)]
#new_valid_covid_fold6=covid[covid['patient_id'].isin(valid_covid_folds[5][0])]
#new_valid_covid_fold7=covid[covid['patient_id'].isin(valid_covid_folds[6][0])]
#new_valid_covid_fold8=covid[covid['patient_id'].isin(valid_covid_folds[7][0])]
#new_valid_covid_fold9=covid[covid['patient_id'].isin(valid_covid_folds[8][0])]
#new_valid_covid_fold10=covid[covid['patient_id'].isin(valid_covid_folds[9][0])]

In [39]:
new_valid_normal_fold1=final_normal_df[final_normal_df['patient_id'].isin(valid_normal_folds1)]
new_valid_normal_fold2=final_normal_df[final_normal_df['patient_id'].isin(valid_normal_folds2)]
new_valid_normal_fold3=final_normal_df[final_normal_df['patient_id'].isin(valid_normal_folds3)]
new_valid_normal_fold4=final_normal_df[final_normal_df['patient_id'].isin(valid_normal_folds4)]
new_valid_normal_fold5=final_normal_df[final_normal_df['patient_id'].isin(valid_normal_folds5)]
#new_valid_covid_fold6=covid[covid['patient_id'].isin(valid_covid_folds[5][0])]
#new_valid_covid_fold7=covid[covid['patient_id'].isin(valid_covid_folds[6][0])]
#new_valid_covid_fold8=covid[covid['patient_id'].isin(valid_covid_folds[7][0])]
#new_valid_covid_fold9=covid[covid['patient_id'].isin(valid_covid_folds[8][0])]
#new_valid_covid_fold10=covid[covid['patient_id'].isin(valid_covid_folds[9][0])]

In [40]:
new_train_covid_fold1=new_train_covid_fold1.reset_index()
new_train_covid_fold1=new_train_covid_fold1.drop(columns=['index'])
new_train_covid_fold2=new_train_covid_fold2.reset_index()
new_train_covid_fold2=new_train_covid_fold2.drop(columns=['index'])
new_train_covid_fold3=new_train_covid_fold3.reset_index()
new_train_covid_fold3=new_train_covid_fold3.drop(columns=['index'])
new_train_covid_fold4=new_train_covid_fold4.reset_index()
new_train_covid_fold4=new_train_covid_fold4.drop(columns=['index'])
new_train_covid_fold5=new_train_covid_fold5.reset_index()
new_train_covid_fold5=new_train_covid_fold5.drop(columns=['index'])
#new_train_covid_fold6=new_train_covid_fold6.reset_index()
#new_train_covid_fold6=new_train_covid_fold6.drop(columns=['index'])
#new_train_covid_fold7=new_train_covid_fold7.reset_index()
#new_train_covid_fold7=new_train_covid_fold7.drop(columns=['index'])
#new_train_covid_fold8=new_train_covid_fold8.reset_index()
#new_train_covid_fold8=new_train_covid_fold8.drop(columns=['index'])
#new_train_covid_fold9=new_train_covid_fold9.reset_index()
#new_train_covid_fold9=new_train_covid_fold9.drop(columns=['index'])
#new_train_covid_fold10=new_train_covid_fold10.reset_index()
#new_train_covid_fold10=new_train_covid_fold10.drop(columns=['index'])

In [41]:
new_train_normal_fold1=new_train_normal_fold1.reset_index()
new_train_normal_fold1=new_train_normal_fold1.drop(columns=['index'])
new_train_normal_fold2=new_train_normal_fold2.reset_index()
new_train_normal_fold2=new_train_normal_fold2.drop(columns=['index'])
new_train_normal_fold3=new_train_normal_fold3.reset_index()
new_train_normal_fold3=new_train_normal_fold3.drop(columns=['index'])
new_train_normal_fold4=new_train_normal_fold4.reset_index()
new_train_normal_fold4=new_train_normal_fold4.drop(columns=['index'])
new_train_normal_fold5=new_train_normal_fold5.reset_index()
new_train_normal_fold5=new_train_normal_fold5.drop(columns=['index'])
#new_train_normal_fold6=new_train_normal_fold6.reset_index()
#new_train_normal_fold6=new_train_normal_fold6.drop(columns=['index'])
#new_train_normal_fold7=new_train_normal_fold7.reset_index()
#new_train_normal_fold7=new_train_normal_fold7.drop(columns=['index'])
#new_train_normal_fold8=new_train_normal_fold8.reset_index()
#new_train_normal_fold8=new_train_normal_fold8.drop(columns=['index'])
#new_train_normal_fold9=new_train_normal_fold9.reset_index()
#new_train_normal_fold9=new_train_normal_fold9.drop(columns=['index'])
#new_train_normal_fold10=new_train_normal_fold10.reset_index()
#new_train_normal_fold10=new_train_normal_fold10.drop(columns=['index'])

In [42]:
new_valid_covid_fold1=new_valid_covid_fold1.reset_index()
new_valid_covid_fold1=new_valid_covid_fold1.drop(columns=['index'])
new_valid_covid_fold2=new_valid_covid_fold2.reset_index()
new_valid_covid_fold2=new_valid_covid_fold2.drop(columns=['index'])
new_valid_covid_fold3=new_valid_covid_fold3.reset_index()
new_valid_covid_fold3=new_valid_covid_fold3.drop(columns=['index'])
new_valid_covid_fold4=new_valid_covid_fold4.reset_index()
new_valid_covid_fold4=new_valid_covid_fold4.drop(columns=['index'])
new_valid_covid_fold5=new_valid_covid_fold5.reset_index()
new_valid_covid_fold5=new_valid_covid_fold5.drop(columns=['index'])
#new_valid_covid_fold6=new_valid_covid_fold6.reset_index()
#new_valid_covid_fold6=new_valid_covid_fold6.drop(columns=['index'])
#new_valid_covid_fold7=new_valid_covid_fold7.reset_index()
#new_valid_covid_fold7=new_valid_covid_fold7.drop(columns=['index'])
#new_valid_covid_fold8=new_valid_covid_fold8.reset_index()
#new_valid_covid_fold8=new_valid_covid_fold8.drop(columns=['index'])
#new_valid_covid_fold9=new_valid_covid_fold9.reset_index()
#new_valid_covid_fold9=new_valid_covid_fold9.drop(columns=['index'])
#new_valid_covid_fold10=new_valid_covid_fold10.reset_index()
#new_valid_covid_fold10=new_valid_covid_fold10.drop(columns=['index'])

In [43]:
new_valid_normal_fold1=new_valid_normal_fold1.reset_index()
new_valid_normal_fold1=new_valid_normal_fold1.drop(columns=['index'])
new_valid_normal_fold2=new_valid_normal_fold2.reset_index()
new_valid_normal_fold2=new_valid_normal_fold2.drop(columns=['index'])
new_valid_normal_fold3=new_valid_normal_fold3.reset_index()
new_valid_normal_fold3=new_valid_normal_fold3.drop(columns=['index'])
new_valid_normal_fold4=new_valid_normal_fold4.reset_index()
new_valid_normal_fold4=new_valid_normal_fold4.drop(columns=['index'])
new_valid_normal_fold5=new_valid_normal_fold5.reset_index()
new_valid_normal_fold5=new_valid_normal_fold5.drop(columns=['index'])
#new_valid_normal_fold6=new_valid_normal_fold6.reset_index()
#new_valid_normal_fold6=new_valid_normal_fold6.drop(columns=['index'])
#new_valid_normal_fold7=new_valid_normal_fold7.reset_index()
#new_valid_normal_fold7=new_valid_normal_fold7.drop(columns=['index'])
#new_valid_normal_fold8=new_valid_normal_fold8.reset_index()
#new_valid_normal_fold8=new_valid_normal_fold8.drop(columns=['index'])
#new_valid_normal_fold9=new_valid_normal_fold9.reset_index()
#new_valid_normal_fold9=new_valid_normal_fold9.drop(columns=['index'])
#new_valid_normal_fold10=new_valid_normal_fold10.reset_index()
#new_valid_normal_fold10=new_valid_normal_fold10.drop(columns=['index'])

In [44]:
print(FOLDNO)

if FOLDNO==1:
    train_fold=pd.concat([new_train_covid_fold1,new_train_normal_fold1])
    valid_fold=pd.concat([new_valid_covid_fold1,new_valid_normal_fold1])
    train_covid_fold=new_train_covid_fold1
    valid_covid_fold=new_valid_covid_fold1
    train_normal_fold=new_train_normal_fold1
    valid_normal_fold=new_valid_normal_fold1
elif FOLDNO==2:
    train_fold=pd.concat([new_train_covid_fold2,new_train_normal_fold2])
    valid_fold=pd.concat([new_valid_covid_fold2,new_valid_normal_fold2])
    train_covid_fold=new_train_covid_fold2
    valid_covid_fold=new_valid_covid_fold2
    train_normal_fold=new_train_normal_fold2
    valid_normal_fold=new_valid_normal_fold2
elif FOLDNO==3:
    train_fold=pd.concat([new_train_covid_fold3,new_train_normal_fold3])
    valid_fold=pd.concat([new_valid_covid_fold3,new_valid_normal_fold3])
    train_covid_fold=new_train_covid_fold3
    valid_covid_fold=new_valid_covid_fold3
    train_normal_fold=new_train_normal_fold3
    valid_normal_fold=new_valid_normal_fold3
elif FOLDNO==4:
    train_fold=pd.concat([new_train_covid_fold4,new_train_normal_fold4])
    valid_fold=pd.concat([new_valid_covid_fold4,new_valid_normal_fold4])
    train_covid_fold=new_train_covid_fold4
    valid_covid_fold=new_valid_covid_fold4
    train_normal_fold=new_train_normal_fold4
    valid_normal_fold=new_valid_normal_fold4
elif FOLDNO==5:
    train_fold=pd.concat([new_train_covid_fold5,new_train_normal_fold5])
    valid_fold=pd.concat([new_valid_covid_fold5,new_valid_normal_fold5])
    train_covid_fold=new_train_covid_fold5
    valid_covid_fold=new_valid_covid_fold5
    train_normal_fold=new_train_normal_fold5
    valid_normal_fold=new_valid_normal_fold5

4


In [45]:
valid_fold=valid_fold.reset_index()
valid_fold=valid_fold.drop(columns=['index'])

In [46]:
val_datagen = ImageDataGenerator(#preprocessing_function=preprocess_input)
                                    rescale=1./255)
                                    #samplewise_center = SAMPLEWISE_CENTER,
                                    #samplewise_std_normalization = SAMPLEWISE_STD_NORM)

In [47]:
valid_covid_fold_cat=pd.DataFrame()
valid_covid_fold_cat['target']=valid_covid_fold['target']
valid_covid_fold_cat['target']=0
valid_normal_fold_cat=pd.DataFrame()
valid_normal_fold_cat['target']=valid_normal_fold['target']
valid_normal_fold_cat['target']=1
valid_fold_cat=pd.concat([valid_covid_fold_cat, valid_normal_fold_cat])
valid_fold_cat_list=valid_fold_cat['target'].to_list()

In [48]:
validation_generator = val_datagen.flow_from_dataframe(valid_fold,
                                                          x_col="image_path",
                                                          y_col="target",
                                                          target_size = (IMG_SIZE, IMG_SIZE),
                                                          #subset = "validation"
                                                          #interpolation="bilinear",
                                                          batch_size = BATCH_SIZE,
                                                          shuffle=False,
                                                          class_mode = CLASS_MODE)

Found 3799 validated image filenames belonging to 2 classes.


In [49]:
os.mkdir('/content/'+MODEL_NAME)
os.mkdir('/content/'+MODEL_NAME+'/comp_orig_predicted_colorbar')
os.mkdir('/content/'+MODEL_NAME+'/comp_heatmap_predicted_colorbar')
os.mkdir('/content/'+MODEL_NAME+'/comp_output_predicted_colorbar')
os.mkdir('/content/'+MODEL_NAME+'/confusion_matrix')
os.mkdir('/content/'+MODEL_NAME+'/loss_figure')
os.mkdir('/content/'+MODEL_NAME+'/accuracy_figure')
os.mkdir('/content/'+MODEL_NAME+'/metrics')
os.mkdir('/content/'+MODEL_NAME+'/models')
os.mkdir('/content/'+MODEL_NAME+'/roc_curve_val')
os.mkdir('/content/'+MODEL_NAME+'/roc_curve_test')
#os.mkdir('/content/'+MODEL_NAME+'/roc_auc_curve')
os.mkdir('/content/'+MODEL_NAME+'/comp_orig_predicted_colorbar_test')
os.mkdir('/content/'+MODEL_NAME+'/comp_heatmap_predicted_colorbar_test')
os.mkdir('/content/'+MODEL_NAME+'/comp_output_predicted_colorbar_test')

In [50]:
model=create_model(IMG_SIZE,MODEL_NAME,CHANNELS,CLASSES,ACTIVATION,WEIGHTS,LEARNING_RATE,CROSSENTROPY,OPTIMIZER_HYPERPARAMS,BETA_1,BETA_2,EPSILON,DECAY,FINE_TUNING,UNFROZEN_LAYERS)

29084464/29084464 [==============================] - 0s 0us/step
Optimizer Hyperparams: False


In [51]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 256, 256, 3)]        0         []                            
                                                                                                  
 zero_padding2d (ZeroPaddin  (None, 262, 262, 3)          0         ['input_1[0][0]']             
 g2D)                                                                                             
                                                                                                  
 conv1/conv (Conv2D)         (None, 128, 128, 64)         9408      ['zero_padding2d[0][0]']      
                                                                                                  
 conv1/bn (BatchNormalizati  (None, 128, 128, 64)         256       ['conv1/conv[0][0]']      

In [52]:
model.load_weights('/content/model_weights/model_'+str(FOLDNO)+'.h5')

In [53]:
val_loss, val_acc = model.evaluate(validation_generator)
print("Validation Accuracy: "+str(val_acc))
print("Validation Loss: "+str(val_loss))

119/119 [==============================] - 41s 164ms/step - loss: 0.1004 - accuracy: 0.9687
Validation Accuracy: 0.968675971031189
Validation Loss: 0.10036468505859375


In [ ]:
valid_tp=0
valid_fn=0
preds_list=[]
preds_not_norm=[]
true_list=[]
for filename in valid_covid_fold['image_path']:
    last_part=filename.rpartition('/')[2]
    orig = cv2.imread(filename, cv2.IMREAD_COLOR)
    resized = cv2.resize(orig, (IMG_SIZE, IMG_SIZE))
    image = resized.astype(np.float64)
    image = np.expand_dims(image, axis=0)
    image = val_datagen.standardize(image)
    preds = model.predict(image)
    i = np.argmax(preds[0])
    true_list.append(0)
    preds_list.append(i)
    preds_not_norm.append([preds[0][0], preds[0][1]])
    if(i==0):
        valid_tp=valid_tp+1
    else:
        valid_fn=valid_fn+1

1/1 [==============================] - 0s 25ms/step


In [ ]:
valid_tn=0
valid_fp=0
for filename in valid_normal_fold['image_path']:
    last_part=filename.rpartition('/')[2]
    orig = cv2.imread(filename, cv2.IMREAD_COLOR)
    resized = cv2.resize(orig, (IMG_SIZE, IMG_SIZE))
    image = resized.astype(np.float64)
    image = np.expand_dims(image, axis=0)
    image = val_datagen.standardize(image)
    preds = model.predict(image)
    i = np.argmax(preds[0])
    true_list.append(1)
    preds_list.append(i)
    preds_not_norm.append([preds[0][0], preds[0][1]])
    if(i==0):
        valid_fp=valid_fp+1
    else:
        valid_tn=valid_tn+1

In [ ]:
cm_true=np.array([[valid_tp, valid_fn] ,
                 [valid_fp ,valid_tn]])

In [ ]:
val_total_cm=[]
val_total_cm.append(cm_true)

In [ ]:
plot_confusion_matrix_test(cm_true,MODEL_NAME, FOLDNO, CLASS1, CLASS2)

In [ ]:
map_dict = {0:1,1:0}
predicted_inverted = list(map(lambda x: map_dict[x],preds_list))

In [ ]:
predicted_notnorm_inverted=[]
for row in preds_not_norm:
    predicted_notnorm_inverted.append([row[1],row[0]])

In [ ]:
true_inverted = list(map(lambda x: map_dict[x],valid_fold_cat['target']))

In [ ]:
predicted_notnorm_inverted_right=[]
for row in predicted_notnorm_inverted:
    predicted_notnorm_inverted_right.append(row[1])

In [ ]:
predicted_notnorm_right=[]
for row in preds_not_norm:
    predicted_notnorm_right.append(row[1])

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, log_loss
accuracy=accuracy_score(true_inverted, predicted_inverted)
loss=log_loss(true_inverted, predicted_notnorm_inverted)
precision=precision_score(true_inverted, predicted_inverted)
recall=recall_score(true_inverted, predicted_inverted)
f1_score=f1_score(true_inverted, predicted_inverted)

In [ ]:
tprs_probs = []
aucs_probs = []
mean_fpr_probs = np.linspace(0,1,100)

fpr_probs, tpr_probs, t_probs = roc_curve(true_inverted, predicted_inverted)
tprs_probs.append(interp(mean_fpr_probs, fpr_probs, tpr_probs))
roc_auc_probs = auc(fpr_probs, tpr_probs)
aucs_probs.append(roc_auc_probs)

with open('/content/'+MODEL_NAME+'/fpr_probs_fold'+str(FOLDNO)+'.txt', 'w') as f:
    for elem in fpr_probs:
        f.write(str(elem))
        f.write('\n')

with open('/content/'+MODEL_NAME+'/tpr_probs_fold'+str(FOLDNO)+'.txt', 'w') as f:
    for elem in tpr_probs:
        f.write(str(elem))
        f.write('\n')

with open('/content/'+MODEL_NAME+'/t_probs_fold'+str(FOLDNO)+'.txt', 'w') as f:
    for elem in t_probs:
        f.write(str(elem))
        f.write('\n')

plot_roc_curve_val_probs(fpr_probs,tpr_probs,FOLDNO,roc_auc_probs,MODEL_NAME)

tprs_labels = []
aucs_labels = []
mean_fpr_labels = np.linspace(0,1,100)

fpr_labels, tpr_labels, t_labels = roc_curve(true_inverted, predicted_notnorm_inverted_right)
tprs_labels.append(interp(mean_fpr_labels, fpr_labels, tpr_labels))
roc_auc_labels = auc(fpr_labels, tpr_labels)
aucs_labels.append(roc_auc_labels)

with open('/content/'+MODEL_NAME+'/fpr_labels_fold'+str(FOLDNO)+'.txt', 'w') as f:
    for elem in fpr_labels:
        f.write(str(elem))
        f.write('\n')

with open('/content/'+MODEL_NAME+'/tpr_labels_fold'+str(FOLDNO)+'.txt', 'w') as f:
    for elem in tpr_labels:
        f.write(str(elem))
        f.write('\n')

with open('/content/'+MODEL_NAME+'/t_labels_fold'+str(FOLDNO)+'.txt', 'w') as f:
    for elem in t_labels:
        f.write(str(elem))
        f.write('\n')

plot_roc_curve_val_labels(fpr_labels,tpr_labels,FOLDNO,roc_auc_labels,MODEL_NAME)

In [ ]:
data1=[]
data1.append([FOLDNO,accuracy,loss,precision,recall,f1_score,val_total_cm[0][0][0],val_total_cm[0][0][1],val_total_cm[0][1][0],val_total_cm[0][1][1]])

validation_metrics = pd.DataFrame(data1, columns = ['Fold','Validation Accuracy', 'Validation Loss', 'Validation Precision', 'Validation Recall', 'Validation F1-Score' ,'[0,0]','[0,1]','[1,0]','[1,1]'])

validation_metrics.to_excel("/content/"+MODEL_NAME+"/metrics/validation_metrics.xlsx",
            sheet_name='Sheet1')

In [ ]:
data2=[]
data2.append([N_SPLITS,IMG_SIZE,CHANNELS,LEARNING_RATE,POOLING,BATCH_SIZE,TEST_SIZE,CLASSES,CLASS1,CLASS2,FINE_TUNING,UNFROZEN_LAYERS,EPOCHS,CLASS_MODE,CROSSENTROPY,MODEL_NAME,OPTIMIZER,OPTIMIZER_HYPERPARAMS,BETA_1,BETA_2,EPSILON,DECAY,ANNEALER,ANNEALER_MONITOR,ANNEALER_FACTOR,ANNEALER_PATIENCE,EARLY,EARLY_MONITOR,EARLY_PATIENCE,RESCALE,SAMPLEWISE_CENTER,SAMPLEWISE_STD_NORM,AUGMENTATION,ROTATION_RANGE,WIDTH_SHIFT_RANGE,HEIGHT_SHIFT_RANGE,ZOOM_RANGE,HORIZONTAL_FLIP,VERTICAL_FLIP])

hyperparameters = pd.DataFrame(data2, columns = ['Number of Folds', 'Image Size', 'Channels','Initial LR','Pooling','Batch Size','Test Size','Classes','Class 1','Class 2','Fine Tuning','Unfrozen Layers','Epochs','Class Mode','Crossentropy','Model Name','Optimizer','Optimizer Hyperparameters','Beta 1','Beta 2','Epsilon','Decay','Implements Annealer','Annealer Monitor','Annealer Factor','Annealer Patience','Implements Early Stopping','Early Stopping Monitor','Early Stopping Patience','Rescale','Samplewise Center','Samplewise Std Normalization','Implements Augmentation','Rotation Range','Width Shift Range','Height Shift Range','Zoom Range','Horizontal Flip','Vertical Flip'])

hyperparameters.to_excel("/content/"+MODEL_NAME+"/metrics/hyperparameters.xlsx",
            sheet_name='Sheet1')

In [ ]:
shutil.make_archive(str(MODEL_NAME)+"_"+str(ROTATION_RANGE)+"_fold_"+str(FOLDNO)+"_"+str(NORMALIZED)+"_"+str(SPLIT)+"_"+str(SEGMENTED)+"_Results", 'zip', "/content/"+str(MODEL_NAME))

In [ ]:
print('AAAA')